In this notebook we parse texts on "Government Policies and Programs" extracted from USGS PDF Reports to construct a structured panel data capturing temporal and spatial policy variations. The unit of analysis in the output panel is `country-year(date)-policy`, with properties including, *inter alia*:

0. policy title with reference number (ref. ID) ([NAME], *str*);
1. administrative level (state/local, *categorical*);
2. issuing authority ([NAME], *str*); 
3. legal hierarchy ([TYPE_OF_LEGAL_DOCUMENT], *str*); 
4. policy domain (export_import_restriction, foreign_investment_access, equity_structure_state_ownership, technological_transfer, domestic_content_requirement, mining_concession, land_concession, tax_royalty, subsidy, labor_standard, environmental_standard, community_interest, foreign_exchange, geodata_secrecy, comprehensive_framework), *mulitiple choices concatenated to a list*);
5. targeted minerals ([NAME], *list*); 
6. contents of the policy ([REGULATORY_TARGET_WIZ_DUTIES], *str*)

Given the lack of training data, there are two *prema facie* technical pathways at our disposal: (1) Purely based on state-of-the-art (SOTA) LLM with few-shot or zero-shot prompt engineering. (2) Hybrid of SOTA-LLM and BERT, where SOTA-LLM is appealed to for generating a training set, before BERT is invoked to accelerate coding process after pre-training.

In [1]:
import sys
sys.path.append("../../modules/text_parsing/")

In [2]:
continent = 'asia'
country_year_policy = 'Indonesia_2019_policy'
txt_path = "../../data/policy_section_txt/usgs_" + continent + '/' + country_year_policy + '.txt'
with open(txt_path, "r", encoding='utf-8') as file:
    input_text = file.read()
print(input_text)

On January 12, 2014, Indonesia’s ban on unprocessed mineral exports went into effect. The ban, Government Regulation (GR) No. 4 of 2009, includes provisions that mineral products should be processed and refined domestically, and that the export of unprocessed minerals is prohibited. Mining companies, however, were allowed to export ore concentrates until January 11, 2017, under the condition that they meet the minimum domestic refining requirements (differs by commodity), pay export duties, and commit to building smelters in the country. On January 11, 2017, the Government of Indonesia issued GR 1/2017, which extended this exemption for the next 5 years to prevent mining companies from reducing their mining activities (PricewaterhouseCoopers Inc., 2019, p. 8–10).
The Ministry of Energy and Mineral Resources (MEMR) stipulated in Regulation No. 9/2017 that foreign owners are required to start divesting their share in stages after the first 5 years of production and could hold a maximum o

In [ ]:
import LLMStructuredOutput as lso
import os
response = lso.qwen_structured(input_text = input_text, # Input text
                      base_url = "https://dashscope.aliyuncs.com/compatible-mode/v1",
                      api_key = os.getenv("DASHSCOPE_API_KEY"), # API-KEY required, drawn from local environment variable
                      model = "qwen3.6-flash")

In [ ]:
import json
response_json = json.loads(response) # Turn the returned string into JSON-schema dictionary
# Re-check
response_json

{'policies': [{'issuing_year': 2009,
   'issuing_month': None,
   'issuing_day': None,
   'document_name': 'Government Regulation (GR) No. 4 of 2009 [GR No. 4 of 2009]',
   'reference_id': 'GR No. 4 of 2009',
   'policy_domain': ['export_import_restriction',
    'domestic_content_requirement'],
   'legal_hierarchy': 'regulation',
   'issuing_authority': None,
   'targeted_mineral': ['minerals'],
   'is_local': None,
   'is_amendment': False},
  {'issuing_year': 2017,
   'issuing_month': 1,
   'issuing_day': 11,
   'document_name': 'GR 1/2017 [GR 1/2017]',
   'reference_id': 'GR 1/2017',
   'policy_domain': ['export_import_restriction',
    'domestic_content_requirement'],
   'legal_hierarchy': 'regulation',
   'issuing_authority': 'Government of Indonesia',
   'targeted_mineral': ['ore concentrates'],
   'is_local': None,
   'is_amendment': True},
  {'issuing_year': 2017,
   'issuing_month': None,
   'issuing_day': None,
   'document_name': 'Regulation No. 9/2017 [Regulation No. 9/2017

In [ ]:
json_path = '../../data/policy_parsing_json/usgs_' + continent + '/' + country_year_policy + '.json' # Output directory
with open(json_path, 'w', encoding='utf-8') as file:
    json.dump(response_json, file, indent=4, ensure_ascii=False)